In [ ]:
# ============ 1. SETTINGS ============
DATA_DIR = "../data/combined"                                   # folder holding classes.json + test/ (or val/)
CKPT_PATH = "../out_combined/vits16-eomt-cityscapes/checkpoints/best.ckpt"

SPLIT = "val"   # falls back to val/valid if this one isn't there
ALPHA = 0.9     # overlay opacity

In [ ]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
print(torch.cuda.device_count())      # → 1
print(torch.cuda.get_device_name(0))  # → name of the *second* physical GPU

In [ ]:
import os

os.environ["LIGHTLY_TRAIN_CACHE_DIR"] = ".cache"
os.environ["LIGHTLY_TRAIN_MODEL_CACHE_DIR"] = ".cache"
os.environ["TORCH_HOME"] = ".cache"

import colorsys
import json
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import lightly_train

Image.MAX_IMAGE_PIXELS = None
ROOT = Path(DATA_DIR)
IGNORE = 255                  # unannotated pixels; drawn transparent
IGNORE_COLOR = (255, 0, 255)

# classes.json maps id -> label, and those ids are what the mask pixels hold.
# Colors are assigned here rather than read from the export, so labels that
# repeat (0 and 1 both "background") still get distinguishable shades. The
# table is indexed by class id, so a class keeps its color in every figure.
CLASSES = {int(k): v for k, v in json.loads((ROOT / "classes.json").read_text()).items()}

def auto_color(n):
    if n < 20:
        return [int(c * 255) for c in plt.get_cmap("tab20")(n)[:3]]
    r, g, b = colorsys.hsv_to_rgb((n * 0.618033988749895) % 1.0, 0.65, 0.95)
    return [int(r * 255), int(g * 255), int(b * 255)]

PALETTE = np.zeros((256, 3), dtype=np.uint8)
for n, i in enumerate(sorted(CLASSES)):
    PALETTE[i] = auto_color(n)
PALETTE[IGNORE] = IGNORE_COLOR

name = lambda i: "ignore/unannotated" if i == IGNORE else CLASSES.get(int(i), f"UNDECLARED {i}")

# read masks raw -- .convert("L") would return palette luminance, not class ids
read_mask = lambda p: (lambda a: a[..., 0] if a.ndim == 3 else a)(np.asarray(Image.open(p)))
EXTS = {".png", ".tif", ".tiff", ".jpg", ".jpeg"}
is_image = lambda p: p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith(".")

TEST = next(ROOT / s for s in [SPLIT, "test", "val", "valid"] if (ROOT / s / "images").is_dir())
IMAGES = sorted(p for p in (TEST / "images").iterdir() if is_image(p))

model = lightly_train.load_model_from_checkpoint(CKPT_PATH)

print(f"dataset    : {ROOT.name}  ({TEST.name} split, {len(IMAGES)} images)")
print(f"checkpoint : {CKPT_PATH}")
print("colors     : fixed for every image")
for i in sorted(CLASSES) + [IGNORE]:
    print("               {:>3}  {:<28} #{:02x}{:02x}{:02x}".format(i, name(i), *PALETTE[i]))

In [ ]:
# ============ 2. RUN INFERENCE AND PLOT ============
def normalize(img, lo=2, hi=98):
    """Percentile stretch so faint tomography slices are actually visible."""
    a, b = np.percentile(img, lo), np.percentile(img, hi)
    return img.astype(np.uint8) if b <= a else (np.clip(img, a, b) - a) / (b - a) * 255


for img_path in IMAGES:
    image = normalize(np.array(Image.open(img_path).convert("RGB"))).astype(np.uint8)

    pred = model.predict(str(img_path))
    pred = pred.detach().cpu().numpy() if hasattr(pred, "detach") else np.asarray(pred)
    pred = pred.astype(np.int64)

    mask_path = TEST / "masks" / img_path.name
    gt = read_mask(mask_path) if is_image(mask_path) else None

    panels = [("image", None), ("prediction", pred)] + ([("ground truth", gt)] if gt is not None else [])
    fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 5.5))

    for ax, (title, overlay) in zip(np.atleast_1d(axes), panels):
        ax.imshow(image)
        if overlay is not None:
            # RGBA so ignore pixels are fully transparent, everything else ALPHA
            rgba = np.dstack([PALETTE[overlay], np.full(overlay.shape, int(255 * ALPHA), np.uint8)])
            rgba[..., 3][overlay == IGNORE] = 0
            ax.imshow(rgba, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")

    ids = sorted(set(np.unique(pred).tolist()) | (set(np.unique(gt).tolist()) if gt is not None else set()))
    np.atleast_1d(axes)[-1].legend(
        handles=[mpatches.Patch(color=PALETTE[i] / 255, label=f"{i}  {name(i)}") for i in ids],
        loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8,
    )

    fig.suptitle(img_path.name, fontsize=11)
    fig.tight_layout()
    plt.show()